# 03 - Matchup-Specific Models with Hyperparameter Tuning



In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer
import xgboost as xgb

pd.set_option("display.float_format", "{:.3f}".format)

DATA_PATH = Path("../data/features_checkpoints.csv")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

C:\Users\bernd\sc2-win-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Step 1 - Load, clean and label matchups

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["winner"])
df["winner"] = df["winner"].astype(int)

# Canonical matchup: alphabetically sort the two race initials
# so PvT == TvP, TvZ == ZvT, etc.
def canonical_matchup(r1, r2):
    return "v".join(sorted([r1[0], r2[0]]))

df["matchup"] = df.apply(lambda r: canonical_matchup(r["p1_race"], r["p2_race"]), axis=1)

games = df.drop_duplicates("replay_file")
print("Games per matchup:")
print(games["matchup"].value_counts())
print(f"\nTotal games : {games['replay_file'].nunique()}")
print(f"Total rows  : {len(df)}")

Games per matchup:
matchup
PvT    3295
TvZ    2710
PvZ    2072
TvT    1340
PvP     931
ZvZ     731
ПvТ       4
Name: count, dtype: int64

Total games : 11083
Total rows  : 84923


---
## Step 2 - Normalize p1/p2 ordering per matchup
For mirror matchups (PvP, TvT, ZvZ) p1/p2 is already arbitrary — symmetry
augmentation handles those.

In [3]:
STAT_COLS = [
    "minerals_current", "vespene_current",
    "minerals_collection_rate", "vespene_collection_rate",
    "workers_active", "army_supply",
    "minerals_total", "vespene_total",
]

TECH_KEYS = [c.replace("p1_","").replace("p2_","") for c in df.columns
             if c.startswith("p1_") and c.endswith("_sec") and "natural" not in c
             and "third" not in c]

ROC_KEYS = ["workers_active_roc", "army_supply_roc", "minerals_collection_rate_roc"]

EVENT_KEYS = ["natural_age", "third_age", "has_natural", "has_third",
              "army_lost_count", "army_lost_per_min", "worker_lost_count"]

EFFICIENCY_KEYS = ["spending_quotient", "supply_block_pct",
                   "weapons_level", "armor_level"]

COMPOSITION_KEYS = [
    "bio_produced", "mech_produced", "air_produced",
    "gateway_produced", "robo_produced", "stargate_produced",
    "ling_bane_produced", "roach_ravager_produced", "hydra_lurker_produced",
    "muta_corr_produced", "ultra_produced", "caster_produced", "queen_produced",
]

OPENING_KEYS = [
    "opening_forge_first", "opening_gate_first", "opening_nexus_first",
    "opening_barracks_first", "opening_cc_first", "opening_pool_first",
    "opening_hatch_first", "opening_ffe", "opening_cannon_rush", "opening_1gate_expand",
]

KEY_UPGRADE_KEYS = [
    "has_stimpack", "has_combat_shield", "has_concussive_shells",
    "has_blink", "has_charge", "has_psi_storm", "has_extended_thermal_lance",
    "has_metabolic_boost", "has_adrenal_glands", "has_glial_reconstitution",
    "has_tunneling_claws",
]

ALL_STAT_KEYS = (STAT_COLS + ROC_KEYS + EVENT_KEYS + EFFICIENCY_KEYS
                 + COMPOSITION_KEYS + OPENING_KEYS + KEY_UPGRADE_KEYS)

def normalize_ordering(df_in):
    """
    For non-mirror matchups, swap p1/p2 so the alphabetically first
    race is always p1. Returns a copy of the DataFrame.
    """
    df_out = df_in.copy()

    p1_cols    = [c for c in df_out.columns if c.startswith("p1_")]
    p2_cols    = [c for c in df_out.columns if c.startswith("p2_")]
    delta_cols = [c for c in df_out.columns if c.startswith("delta_")]

    needs_flip = (df_out["p1_race"].str[0] > df_out["p2_race"].str[0])

    tmp = df_out.loc[needs_flip, p1_cols].values.copy()
    df_out.loc[needs_flip, p1_cols] = df_out.loc[needs_flip, p2_cols].values
    df_out.loc[needs_flip, p2_cols] = tmp
    df_out.loc[needs_flip, delta_cols] = -df_out.loc[needs_flip, delta_cols]
    df_out.loc[needs_flip, "winner"] = 3 - df_out.loc[needs_flip, "winner"]

    print(f"Flipped {needs_flip.sum()} rows to normalize p1/p2 ordering")
    return df_out

df = normalize_ordering(df)

Flipped 30637 rows to normalize p1/p2 ordering


---
## Step 3 - Build feature matrix and shared utilities

In [4]:
# Encode races
le = LabelEncoder()
le.fit(pd.concat([df["p1_race"], df["p2_race"]]).unique())
df["p1_race_enc"] = le.transform(df["p1_race"])
df["p2_race_enc"] = le.transform(df["p2_race"])
print("Race encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

feature_cols = (
    ["game_minute", "p1_race_enc", "p2_race_enc"] +
    [f"p1_{c}" for c in ALL_STAT_KEYS] +
    [f"p2_{c}" for c in ALL_STAT_KEYS] +
    [f"delta_{c}" for c in ALL_STAT_KEYS]
)
# Keep only columns that exist in the dataframe, and deduplicate (STAT_COLS
# already contains minerals_total/vespene_total so ALL_STAT_KEYS had them twice)
feature_cols = list(dict.fromkeys(c for c in feature_cols if c in df.columns))
print(f"Feature columns: {len(feature_cols)}")

#  no duplicates
assert len(feature_cols) == len(set(feature_cols)), "Duplicate feature columns detected!"

Race encoding: {'Protoss': np.int64(0), 'Terran': np.int64(1), 'Zerg': np.int64(2), 'Протоссы': np.int64(3), 'Терраны': np.int64(4)}
Feature columns: 161


In [5]:
def augment_flip(X, y):
    """
    Swap p1/p2, negate deltas, flip label.
    Works at numpy level to avoid pandas column alignment issues.
    """
    cols  = list(X.columns)
    arr   = X.values.copy()

    # Swap each p1_* column with its matching p2_* column
    for i, col in enumerate(cols):
        if col.startswith("p1_"):
            partner = col.replace("p1_", "p2_")
            if partner in cols:
                j = cols.index(partner)
                arr[:, i], arr[:, j] = arr[:, j].copy(), arr[:, i].copy()

    # Negate delta columns
    for i, col in enumerate(cols):
        if col.startswith("delta_"):
            arr[:, i] = -arr[:, i]

    X_flip = pd.DataFrame(arr, columns=cols)
    y_flip = (1 - y.reset_index(drop=True)).reset_index(drop=True)
    return X_flip, y_flip


def prepare_matchup(df_match, test_size=0.15, random_state=42):
    """
    Split a matchup DataFrame into train/val/test at the game level,
    impute, and augment training data.
    """
    groups       = df_match["replay_file"].reset_index(drop=True)
    unique_games = groups.unique()

    train_games, test_games = train_test_split(
        unique_games, test_size=test_size, random_state=random_state)
    train_games_inner, val_games = train_test_split(
        train_games, test_size=0.15, random_state=random_state)

    def mask(games): return groups.isin(games)

    X_all = df_match[feature_cols]
    y_all = (df_match["winner"] == 1).astype(int)

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(
        imputer.fit_transform(X_all),
        columns=feature_cols
    )  # clean integer index
    y_all = y_all.reset_index(drop=True)

    X_inner = X_imp[mask(train_games_inner)].reset_index(drop=True)
    y_inner = y_all[mask(train_games_inner)].reset_index(drop=True)
    X_val   = X_imp[mask(val_games)].reset_index(drop=True)
    y_val   = y_all[mask(val_games)].reset_index(drop=True)
    X_test  = X_imp[mask(test_games)].reset_index(drop=True)
    y_test  = y_all[mask(test_games)].reset_index(drop=True)

    X_flip, y_flip = augment_flip(X_inner, y_inner)
    X_tr = pd.concat([X_inner, X_flip], ignore_index=True)
    y_tr = pd.concat([y_inner, y_flip], ignore_index=True)

    groups_train     = groups[mask(train_games)].reset_index(drop=True)
    groups_train_aug = pd.concat([groups_train, groups_train], ignore_index=True)

    return X_tr, y_tr, X_val, y_val, X_test, y_test, imputer, groups_train_aug


---
## Step 4 - Optuna hyperparameter tuning


In [6]:
def tune_xgb(X_tr, y_tr, X_val, y_val, n_trials=100, random_state=42):
    """Run Optuna to find best XGBoost hyperparameters."""
    # Convert everything to numpy — XGBoost 3.x is strict about types
    X_tr_arr  = np.asarray(X_tr,  dtype=np.float32)
    X_val_arr = np.asarray(X_val, dtype=np.float32)
    y_tr_arr  = np.asarray(y_tr,  dtype=np.float32).ravel()
    y_val_arr = np.asarray(y_val, dtype=np.float32).ravel()

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth":         trial.suggest_int("max_depth", 3, 8),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
            "gamma":             trial.suggest_float("gamma", 0.0, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
        m = xgb.XGBClassifier(
            **params, eval_metric="logloss",
            random_state=random_state, verbosity=0,
            early_stopping_rounds=20,
        )
        m.fit(X_tr_arr, y_tr_arr, eval_set=[(X_val_arr, y_val_arr)], verbose=False)
        return roc_auc_score(y_val_arr, m.predict_proba(X_val_arr)[:, 1])

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value

---
## Step 5 - Train matchup-specific models

We train PvT, TvZ, TvT, PvZ, PvP individually.
ZvZ falls back to a global model (only 47 games).

In [7]:
MATCHUPS_TO_TRAIN = ["PvT", "TvZ", "TvT", "PvZ", "PvP"]

matchup_models  = {}
matchup_results = []

for matchup in MATCHUPS_TO_TRAIN:
    df_m = df[df["matchup"] == matchup]
    n_games = df_m["replay_file"].nunique()
    sep = "=" * 50
    print()
    print(sep)
    print(f"{matchup}  ({n_games} games, {len(df_m)} rows)")
    print(sep)

    X_tr, y_tr, X_val, y_val, X_test, y_test, imp, _ = prepare_matchup(df_m)
    print(f"  Train: {len(X_tr)} rows (aug)   Val: {len(X_val)}   Test: {len(X_test)}")

    print(f"  Tuning (100 Optuna trials)...")
    best_params, best_auc = tune_xgb(X_tr, y_tr, X_val, y_val, n_trials=100)
    print(f"  Best val AUC: {best_auc:.4f}")

    # Final model on train+val combined
    unique_games = df_m["replay_file"].unique()
    train_games, test_games = train_test_split(unique_games, test_size=0.15, random_state=42)
    df_tv = df_m[df_m["replay_file"].isin(train_games)]

    imp_final = SimpleImputer(strategy="median")
    X_tv_all = pd.DataFrame(imp_final.fit_transform(df_tv[feature_cols]), columns=feature_cols)
    y_tv_all = (df_tv["winner"] == 1).astype(int).reset_index(drop=True)

    X_flip, y_flip = augment_flip(X_tv_all, y_tv_all)
    X_tv_aug = np.asarray(pd.concat([X_tv_all, X_flip], ignore_index=True), dtype=np.float32)
    y_tv_aug = np.asarray(pd.concat([y_tv_all, y_flip], ignore_index=True), dtype=np.float32).ravel()

    final = xgb.XGBClassifier(**best_params, eval_metric="logloss", random_state=42, verbosity=0)
    final.fit(X_tv_aug, y_tv_aug, verbose=False)

    # Evaluate on test set
    df_test_m = df_m[df_m["replay_file"].isin(test_games)]
    X_test_imp = np.asarray(imp_final.transform(df_test_m[feature_cols]), dtype=np.float32)
    y_test_m   = np.asarray((df_test_m["winner"] == 1).astype(int), dtype=np.float32).ravel()

    acc = accuracy_score(y_test_m, final.predict(X_test_imp))
    auc = roc_auc_score(y_test_m, final.predict_proba(X_test_imp)[:, 1])
    print(f"  Test accuracy: {acc:.3f}   AUC: {auc:.3f}")
    print(classification_report(y_test_m, final.predict(X_test_imp),
                                  target_names=["P2 wins", "P1 wins"]))

    matchup_models[matchup] = {"model": final, "imputer": imp_final,
                                "feature_cols": feature_cols, "best_params": best_params}
    matchup_results.append({"matchup": matchup, "n_games": n_games,
                             "test_acc": acc, "test_auc": auc})


PvT  (3295 games, 26575 rows)
  Train: 38020 rows (aug)   Val: 3530   Test: 4035
  Tuning (100 Optuna trials)...
  Best val AUC: 0.6533
  Test accuracy: 0.627   AUC: 0.691
              precision    recall  f1-score   support

     P2 wins       0.60      0.53      0.57      1846
     P1 wins       0.64      0.71      0.67      2189

    accuracy                           0.63      4035
   macro avg       0.62      0.62      0.62      4035
weighted avg       0.62      0.63      0.62      4035


TvZ  (2710 games, 23356 rows)
  Train: 33514 rows (aug)   Val: 3035   Test: 3564
  Tuning (100 Optuna trials)...
  Best val AUC: 0.6782
  Test accuracy: 0.614   AUC: 0.687
              precision    recall  f1-score   support

     P2 wins       0.60      0.77      0.67      1834
     P1 wins       0.65      0.44      0.53      1730

    accuracy                           0.61      3564
   macro avg       0.62      0.61      0.60      3564
weighted avg       0.62      0.61      0.60      3564



---
## Step 6 - Train global fallback model (for ZvZ)

In [8]:
print("Training global fallback model...")
X_tr_g, y_tr_g, X_val_g, y_val_g, X_test_g, y_test_g, imp_g, _ = prepare_matchup(df)

best_params_g, best_auc_g = tune_xgb(X_tr_g, y_tr_g, X_val_g, y_val_g, n_trials=100)
print(f"Global best val AUC: {best_auc_g:.4f}")

global_model = xgb.XGBClassifier(
    **best_params_g, eval_metric="logloss", random_state=42, verbosity=0)
global_model.fit(
    np.asarray(X_tr_g, dtype=np.float32),
    np.asarray(y_tr_g, dtype=np.float32).ravel(),
    verbose=False,
)

X_test_g_arr = np.asarray(X_test_g, dtype=np.float32)
y_test_g_arr = np.asarray(y_test_g, dtype=np.float32).ravel()
acc_g = accuracy_score(y_test_g_arr, global_model.predict(X_test_g_arr))
auc_g = roc_auc_score(y_test_g_arr, global_model.predict_proba(X_test_g_arr)[:, 1])
print(f"Global test accuracy: {acc_g:.3f}   AUC: {auc_g:.3f}")
matchup_results.append({"matchup": "global", "n_games": df["replay_file"].nunique(),
                         "test_acc": acc_g, "test_auc": auc_g})

Training global fallback model...
Global best val AUC: 0.6771
Global test accuracy: 0.639   AUC: 0.704


---
## Step 7 - Compare all matchup models

In [9]:
results_df = pd.DataFrame(matchup_results).sort_values("test_auc", ascending=False)
print(results_df.to_string(index=False))

fig = px.bar(
    results_df[results_df["matchup"] != "global"],
    x="matchup", y="test_acc", text="test_acc",
    title="Test accuracy per matchup model",
    labels={"matchup": "Matchup", "test_acc": "Accuracy"},
    color="test_acc", color_continuous_scale="Blues",
    range_y=[0.5, 0.9],
)
fig.add_hline(y=acc_g, line_dash="dash", line_color="red",
              annotation_text=f"Global model ({acc_g:.3f})")
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.show()

matchup  n_games  test_acc  test_auc
 global    11083     0.639     0.704
    PvT     3295     0.627     0.691
    TvZ     2710     0.614     0.687
    PvP      931     0.627     0.679
    TvT     1340     0.616     0.646
    PvZ     2072     0.621     0.643


---
## Step 8 - Accuracy by game minute per matchup

In [10]:
fig = go.Figure()

for matchup in MATCHUPS_TO_TRAIN:
    df_m = df[df["matchup"] == matchup]
    info = matchup_models[matchup]

    # Use only held-out test games (same split as training)
    unique_games = df_m["replay_file"].unique()
    _, test_games = train_test_split(unique_games, test_size=0.15, random_state=42)
    df_test = df_m[df_m["replay_file"].isin(test_games)].copy()

    X_test = np.asarray(info["imputer"].transform(df_test[feature_cols]), dtype=np.float32)
    preds  = info["model"].predict(X_test)
    truth  = (df_test["winner"] == 1).astype(int).values

    df_test = df_test.reset_index(drop=True)
    df_test["correct"] = (preds == truth).astype(int)

    acc_by_min = df_test.groupby("game_minute")["correct"].mean()
    fig.add_trace(go.Scatter(
        x=acc_by_min.index, y=acc_by_min.values,
        mode="lines+markers", name=matchup,
    ))

fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="Random")
fig.update_layout(
    title="Accuracy by game minute — per matchup model (test set only)",
    xaxis_title="Game minute", yaxis_title="Accuracy",
    yaxis_range=[0.4, 1.0], height=450,
)
fig.show()

---
## Step 9 - Save all models

In [11]:
# Save each matchup model + its imputer
for matchup, info in matchup_models.items():
    joblib.dump(info["model"],   MODEL_DIR / f"xgb_{matchup}.pkl")
    joblib.dump(info["imputer"], MODEL_DIR / f"imputer_{matchup}.pkl")
    print(f"Saved {matchup} model")

# Save global fallback
joblib.dump(global_model, MODEL_DIR / "xgb_global.pkl")
joblib.dump(imp_g,        MODEL_DIR / "imputer_global.pkl")

# Save shared artifacts
joblib.dump(feature_cols, MODEL_DIR / "feature_cols.pkl")
joblib.dump(le,           MODEL_DIR / "label_encoder.pkl")

print("\nAll models saved to", MODEL_DIR)
print("\nModel sizes:")
for f in sorted(MODEL_DIR.glob("*.pkl")):
    print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved PvT model
Saved TvZ model
Saved TvT model
Saved PvZ model
Saved PvP model

All models saved to ..\models

Model sizes:
  feature_cols.pkl                          3.6 KB
  imputer.pkl                               1.5 KB
  imputer_global.pkl                        5.6 KB
  imputer_PvP.pkl                           5.6 KB
  imputer_PvT.pkl                           5.6 KB
  imputer_PvZ.pkl                           5.6 KB
  imputer_TvT.pkl                           5.6 KB
  imputer_TvZ.pkl                           5.6 KB
  label_encoder.pkl                         0.5 KB
  xgb_global.pkl                            5790.4 KB
  xgb_PvP.pkl                               843.7 KB
  xgb_PvT.pkl                               2790.8 KB
  xgb_PvZ.pkl                               1767.6 KB
  xgb_TvT.pkl                               3955.3 KB
  xgb_TvZ.pkl                               852.0 KB
  xgb_win_predictor.pkl                     1854.8 KB


In [12]:
import os
print(os.cpu_count())

12
